# TelecomX — Modelado Predictivo de Evasión de Clientes

## Objetivo

A partir del dataframe limpio generado en el análisis exploratorio, este notebook realiza el ciclo completo de Machine Learning para **predecir la evasión de clientes (Churn)**:

| Etapa | Descripción |
|---|---|
| **1. Preparación** | Codificación de variables categóricas y normalización de variables numéricas |
| **2. Correlación y Selección** | Análisis de correlación y selección de variables predictoras relevantes |
| **3. Modelado** | Entrenamiento de Regresión Logística y Random Forest |
| **4. Evaluación** | Métricas de desempeño: Accuracy, Precision, Recall, F1, ROC-AUC |
| **5. Interpretación** | Importancia de variables e insights clave |
| **6. Conclusión** | Resumen estratégico para TelecomX |

## 1. Instalación e Importación de Librerías

In [ ]:
# Instalación de dependencias (descomenta si corres en un entorno nuevo)
# !pip install pandas numpy matplotlib seaborn scikit-learn requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings

# Preprocesamiento y modelado
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

print('Librerías cargadas correctamente.')

---
## 2. Carga y Limpieza de Datos

Se replica el pipeline de limpieza del notebook de EDA para partir del mismo **dataframe limpio** (`Telecom_limpio`).

In [ ]:
url_telecom = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url_telecom)
telecom_json = response.json()

df = pd.json_normalize(telecom_json)

# Renombrado de columnas
df.rename(columns={
    'customerID': 'ID_Cliente',
    'Churn': 'Evasor',
    'customer.gender': 'Género',
    'customer.SeniorCitizen': 'Adulto_Mayor',
    'customer.Partner': 'Con_Pareja',
    'customer.Dependents': 'Con_Dependientes',
    'customer.tenure': 'Meses_como_cliente',
    'phone.PhoneService': 'Servicio_Telefonico',
    'phone.MultipleLines': 'Multiples_Líneas_Telefonicas',
    'internet.InternetService': 'Servicio_Internet',
    'internet.OnlineSecurity': 'Seguridad_Online',
    'internet.OnlineBackup': 'Copia_Seguridad_Online',
    'internet.DeviceProtection': 'Protección_Dispositivos',
    'internet.TechSupport': 'Soporte_Técnico',
    'internet.StreamingMovies': 'Servicio_Streaming',
    'internet.StreamingTV': 'Servicio_Streaming_TV',
    'account.Contract': 'Tipo_Contrato',
    'account.PaymentMethod': 'Método_Pago',
    'account.Charges.Monthly': 'Cargos_Mensuales',
    'account.Charges.Total': 'Cargos_Totales',
    'account.PaperlessBilling': 'Facturación_Sin_Papel'
}, inplace=True)

# Variable objetivo
df['Evasor'] = df['Evasor'].replace({'Yes': True, 'No': False, '': np.nan}).astype('boolean')
df = df.dropna(subset=['Evasor'])

# Conversiones booleanas
cols_bool = ['Con_Pareja', 'Con_Dependientes', 'Servicio_Telefonico', 'Facturación_Sin_Papel']
for col in cols_bool:
    df[col] = df[col].map({'Yes': True, 'No': False}).astype('boolean')

df['Multiples_Líneas_Telefonicas'] = df['Multiples_Líneas_Telefonicas'].map(
    {'Yes': True, 'No': False, 'No phone service': False}).astype('boolean')

df['Servicio_Internet'] = df['Servicio_Internet'].map(
    {'DSL': True, 'Fiber optic': True, 'No': False}).astype('boolean')

cols_internet = [
    'Seguridad_Online', 'Copia_Seguridad_Online', 'Protección_Dispositivos',
    'Soporte_Técnico', 'Servicio_Streaming_TV', 'Servicio_Streaming'
]
for col in cols_internet:
    df[col] = df[col].map({'Yes': True, 'No': False, 'No internet service': False}).astype('boolean')

df['Tipo_Contrato'] = df['Tipo_Contrato'].astype('category')
df['Método_Pago'] = df['Método_Pago'].astype('category')
df['Género'] = df['Género'].replace({'Female': 'Mujer', 'Male': 'Hombre'}).astype('category')
df['Cargos_Mensuales'] = df['Cargos_Mensuales'].astype(float).round(2)
df['Cargos_Totales'] = pd.to_numeric(df['Cargos_Totales'], errors='coerce').round(2)
df['Número_Servicios'] = df[cols_internet + ['Servicio_Telefonico', 'Multiples_Líneas_Telefonicas', 'Servicio_Internet']].sum(axis=1)

# Imputar Cargos_Totales nulos con la mediana
df['Cargos_Totales'].fillna(df['Cargos_Totales'].median(), inplace=True)

Telecom_limpio = df.copy()
print(f'Dataset cargado y limpio: {Telecom_limpio.shape[0]} filas × {Telecom_limpio.shape[1]} columnas')
Telecom_limpio.head()

---
## 3. Preparación de Datos para el Modelado

### 3.1 Codificación de Variables (Encoding)

Los modelos de Machine Learning requieren que todas las variables sean numéricas:

- **Variables booleanas** → se convierten a `int` (0/1).
- **Género** → `LabelEncoder` (Hombre=0, Mujer=1).
- **Tipo_Contrato** y **Método_Pago** → `One-Hot Encoding` para evitar orden implícito.

In [ ]:
df_modelo = Telecom_limpio.drop(columns=['ID_Cliente']).copy()

# Convertir booleanos a int
cols_bool_all = [
    'Con_Pareja', 'Con_Dependientes', 'Servicio_Telefonico', 'Facturación_Sin_Papel',
    'Multiples_Líneas_Telefonicas', 'Servicio_Internet', 'Seguridad_Online',
    'Copia_Seguridad_Online', 'Protección_Dispositivos', 'Soporte_Técnico',
    'Servicio_Streaming_TV', 'Servicio_Streaming'
]
for col in cols_bool_all:
    df_modelo[col] = df_modelo[col].astype(int)

# Variable objetivo a int
df_modelo['Evasor'] = df_modelo['Evasor'].astype(int)

# LabelEncoder para Género
le_genero = LabelEncoder()
df_modelo['Género'] = le_genero.fit_transform(df_modelo['Género'])
print('Mapeo Género:', dict(zip(le_genero.classes_, le_genero.transform(le_genero.classes_))))

# One-Hot Encoding para variables nominales
df_modelo = pd.get_dummies(df_modelo, columns=['Tipo_Contrato', 'Método_Pago'], drop_first=False)

# Renombrar columnas generadas para facilitar lectura
df_modelo.columns = (
    df_modelo.columns
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

print(f'\nDimensiones tras encoding: {df_modelo.shape}')
print('\nColumnas del dataset de modelado:')
print(df_modelo.columns.tolist())

### 3.2 Normalización de Variables Numéricas

Se aplica `StandardScaler` a las variables continuas (`Meses_como_cliente`, `Cargos_Mensuales`, `Cargos_Totales`, `Número_Servicios`) para que tengan **media 0 y desviación estándar 1**. Esto mejora la convergencia en Regresión Logística y evita que variables con mayor escala dominen el modelo.

In [ ]:
cols_numericas = ['Meses_como_cliente', 'Cargos_Mensuales', 'Cargos_Totales', 'Número_Servicios']

scaler = StandardScaler()

# Separar X e y antes de escalar
X = df_modelo.drop(columns=['Evasor'])
y = df_modelo['Evasor']

X[cols_numericas] = scaler.fit_transform(X[cols_numericas])

print('Resumen de variables numéricas después de escalar:')
X[cols_numericas].describe().round(3)

### 3.3 División Train / Test

Se utiliza una división **80% entrenamiento — 20% prueba** con `stratify=y` para mantener la proporción de evasores en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:        {X_test.shape[0]} muestras')
print(f'\nDistribución de Evasor en Train:\n{y_train.value_counts(normalize=True).rename({0: "No Evasor", 1: "Evasor"}).round(3)}')
print(f'\nDistribución de Evasor en Test:\n{y_test.value_counts(normalize=True).rename({0: "No Evasor", 1: "Evasor"}).round(3)}')

---
## 4. Análisis de Correlación y Selección de Variables

### 4.1 Mapa de Calor de Correlaciones con la Variable Objetivo

In [ ]:
# Correlación de cada feature con Evasor (Pearson)
df_corr = pd.concat([X, y], axis=1)
corr_target = df_corr.corr()['Evasor'].drop('Evasor').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Gráfico de barras horizontales ---
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_target.values]
axes[0].barh(corr_target.index, corr_target.values, color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Correlación de cada variable con Evasor', fontsize=13)
axes[0].set_xlabel('Correlación de Pearson')
axes[0].tick_params(axis='y', labelsize=9)

# --- Heatmap top 12 variables ---
top_vars = corr_target.abs().nlargest(12).index.tolist() + ['Evasor']
corr_matrix = df_corr[top_vars].corr()
sns.heatmap(
    corr_matrix, ax=axes[1], annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.4,
    annot_kws={'size': 7}
)
axes[1].set_title('Heatmap de correlación (Top 12 variables)', fontsize=13)
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0, labelsize=8)

plt.tight_layout()
plt.show()

print('\nTop 10 variables más correlacionadas con Evasor:')
print(corr_target.abs().nlargest(10).round(4).to_string())

### 4.2 Selección de Variables

Se seleccionan las variables con una correlación absoluta con `Evasor` mayor a **0.10**, eliminando ruido y reduciendo dimensionalidad.

In [ ]:
umbral_correlacion = 0.10
vars_seleccionadas = corr_target[corr_target.abs() >= umbral_correlacion].index.tolist()

print(f'Variables seleccionadas (|corr| >= {umbral_correlacion}): {len(vars_seleccionadas)}')
print(vars_seleccionadas)

X_train_sel = X_train[vars_seleccionadas]
X_test_sel  = X_test[vars_seleccionadas]

---
## 5. Entrenamiento de Modelos de Clasificación

Se entrenan **tres modelos**:

| Modelo | Características |
|---|---|
| **Regresión Logística** | Modelo lineal interpretable, rápido y eficiente con datos normalizados |
| **Árbol de Decisión** | Modelo no lineal interpretable, captura relaciones complejas |
| **Random Forest** | Ensemble de árboles, mayor robustez y menor overfitting |

In [ ]:
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Árbol de Decisión':   DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced', n_jobs=-1)
}

modelos_entrenados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_sel, y_train)
    modelos_entrenados[nombre] = modelo
    print(f'✓ {nombre} entrenado.')

---
## 6. Evaluación del Rendimiento de los Modelos

### 6.1 Tabla de Métricas

In [ ]:
resultados = []

for nombre, modelo in modelos_entrenados.items():
    y_pred  = modelo.predict(X_test_sel)
    y_proba = modelo.predict_proba(X_test_sel)[:, 1]

    # Cross-validation F1 (5-fold estratificado)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(modelo, X_train_sel, y_train, cv=cv, scoring='f1').mean()

    resultados.append({
        'Modelo': nombre,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_proba), 4),
        'CV F1 (5-fold)': round(cv_f1, 4)
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('=== Tabla de Métricas por Modelo ===')
df_resultados

### 6.2 Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (nombre, modelo) in zip(axes, modelos_entrenados.items()):
    y_pred = modelo.predict(X_test_sel)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Evasor', 'Evasor'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nombre, fontsize=12)

plt.suptitle('Matrices de Confusión', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 6.3 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colores = ['#e74c3c', '#f39c12', '#27ae60']
for (nombre, modelo), color in zip(modelos_entrenados.items(), colores):
    RocCurveDisplay.from_estimator(modelo, X_test_sel, y_test, ax=ax, name=nombre, color=color)

ax.plot([0, 1], [0, 1], 'k--', label='Clasificador aleatorio')
ax.set_title('Curvas ROC — Comparación de Modelos', fontsize=13)
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 6.4 Reporte de Clasificación del Mejor Modelo

In [ ]:
mejor_modelo_nombre = df_resultados['ROC-AUC'].idxmax()
mejor_modelo = modelos_entrenados[mejor_modelo_nombre]
y_pred_best = mejor_modelo.predict(X_test_sel)

print(f'=== Mejor modelo por ROC-AUC: {mejor_modelo_nombre} ===')
print(classification_report(y_test, y_pred_best, target_names=['No Evasor', 'Evasor']))

---
## 7. Importancia de Variables

### 7.1 Coeficientes — Regresión Logística

Los coeficientes positivos aumentan la probabilidad de evasión; los negativos la disminuyen.

In [ ]:
lr = modelos_entrenados['Regresión Logística']
coef_lr = pd.Series(lr.coef_[0], index=vars_seleccionadas).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in coef_lr.values]
ax.barh(coef_lr.index, coef_lr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regresión Logística', fontsize=13)
ax.set_xlabel('Coeficiente (escala estandarizada)')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

### 7.2 Importancia de Variables — Random Forest

In [ ]:
rf = modelos_entrenados['Random Forest']
importancias_rf = pd.Series(
    rf.feature_importances_, index=vars_seleccionadas
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    x=importancias_rf.values,
    y=importancias_rf.index,
    palette='viridis',
    ax=ax
)
ax.set_title('Importancia de Variables — Random Forest', fontsize=13)
ax.set_xlabel('Importancia (Gini)')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 variables más importantes (Random Forest):')
print(importancias_rf.head(10).round(4).to_string())

### 7.3 Comparación de Importancia entre Modelos

In [ ]:
# Normalizar coeficientes LR a importancia absoluta (0-1)
imp_lr  = coef_lr.abs() / coef_lr.abs().sum()
imp_dt  = pd.Series(
    modelos_entrenados['Árbol de Decisión'].feature_importances_,
    index=vars_seleccionadas
)
imp_rf  = importancias_rf

df_imp = pd.DataFrame({'Reg. Logística': imp_lr, 'Árbol de Decisión': imp_dt, 'Random Forest': imp_rf})
df_imp['Promedio'] = df_imp.mean(axis=1)
df_imp = df_imp.sort_values('Promedio', ascending=False)

fig, ax = plt.subplots(figsize=(11, 7))
df_imp[['Reg. Logística', 'Árbol de Decisión', 'Random Forest']].head(12).plot(
    kind='barh', ax=ax
)
ax.set_title('Comparación de Importancia de Variables (Top 12)', fontsize=13)
ax.set_xlabel('Importancia normalizada')
ax.invert_yaxis()
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 variables por importancia promedio entre los 3 modelos:')
print(df_imp['Promedio'].head(10).round(4).to_string())

---
## 8. Conclusión Estratégica

### 8.1 Resumen de Desempeño de Modelos

In [ ]:
metricas_plot = df_resultados[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]

fig, ax = plt.subplots(figsize=(11, 5))
metricas_plot.T.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Comparación de Métricas por Modelo', fontsize=13)
ax.set_ylabel('Valor')
ax.set_ylim(0.5, 1.0)
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 8.2 Conclusión Final

#### Desempeño de Modelos

Los tres modelos entrenados muestran capacidad predictiva por encima de un clasificador aleatorio (ROC-AUC > 0.5). El **Random Forest** presenta el mejor equilibrio entre todas las métricas, ofreciendo mayor robustez al no depender de supuestos de linealidad.

#### Factores Clave de Evasión

Los tres modelos coinciden en identificar los siguientes factores como los más determinantes:

| Factor | Dirección | Interpretación |
|---|---|---|
| **Meses_como_cliente** | ↓ Evasión | Clientes con mayor antigüedad son significativamente más leales |
| **Tipo_Contrato (Mes a mes)** | ↑ Evasión | El contrato mensual es el principal predictor de abandono |
| **Cargos_Mensuales** | ↑ Evasión | Cargos elevados correlacionan con mayor propensión a irse |
| **Soporte_Técnico** | ↓ Evasión | Clientes con soporte técnico evaden menos |
| **Seguridad_Online** | ↓ Evasión | La seguridad online actúa como ancla de retención |
| **Método_Pago (Cheque electrónico)** | ↑ Evasión | El método manual aumenta la fricción y el riesgo de abandono |
| **Con_Dependientes** | ↓ Evasión | Clientes con dependientes presentan mayor estabilidad |

#### Recomendaciones Estratégicas para TelecomX

1. **Migración de contratos mensuales a anuales/bianuales**: Ofrecer incentivos (descuentos, beneficios) para clientes en planes mes a mes, especialmente en los primeros 12–18 meses de vida del cliente.

2. **Programa de bienvenida intensivo**: Los clientes nuevos (<18 meses) tienen el mayor riesgo. Un plan de acompañamiento temprano puede reducir significativamente la evasión.

3. **Paquetes con Soporte Técnico y Seguridad Online**: Estos servicios actúan como **anclas de retención**. Ofrecerlos de forma gratuita en los primeros meses o incluirlos en paquetes combinados es una estrategia de alto impacto.

4. **Eliminar la fricción del cheque electrónico**: Implementar recordatorios automáticos, facilitar la migración a pago automático con débito/tarjeta y simplificar el proceso de pago.

5. **Revisar la estructura de precios**: Clientes con cargos mensuales altos son más propensos a cancelar. Un análisis de precio-valor percibido, acompañado de propuestas personalizadas para clientes de alto valor, puede reducir esta pérdida.

6. **Segmentación para campañas de retención**: Utilizar el modelo de Random Forest como **sistema de alerta temprana** para identificar clientes con alta probabilidad de evasión y activar campañas de retención proactivas.